# Stage 5 — mobility forensic analysis

This notebook analyzes cell-tower behavior. The dataset provides cell IDs but not coordinates, so the analysis measures tower diversity and transitions, not physical distance or route geometry.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED=Path('../outputs/processed/all_naive_12pct_200sims')
OUTPUT=Path('../outputs/mobility/all_naive_12pct_200sims')
OUTPUT.mkdir(parents=True,exist_ok=True)
cdr=pd.read_csv(PROCESSED/'canonical_cdr.csv',dtype={'src_number':'string','dst_number':'string'},parse_dates=['timestamp'])
fraud=set(pd.read_csv(PROCESSED/'fraudulent_users.csv',dtype={'src_number':'string'})['src_number'].dropna().tolist())
print('canonical CDRs:',len(cdr),'fraud labels:',len(fraud))

canonical CDRs: 86873 fraud labels: 200


C:\Users\salai_wciilqo\AppData\Local\Temp\ipykernel_32500\2150937125.py:8: DtypeWarning: Columns (3: src_imei, 7: dst_imei) have mixed types. Specify dtype option on import or set low_memory=False.
  cdr=pd.read_csv(PROCESSED/'canonical_cdr.csv',dtype={'src_number':'string','dst_number':'string'},parse_dates=['timestamp'])


In [2]:
source=cdr[['record_id','timestamp','src_number','initial_src_cell_id','current_src_cell_id']].rename(columns={'src_number':'sim','initial_src_cell_id':'initial_cell','current_src_cell_id':'current_cell'})
destination=cdr[['record_id','timestamp','dst_number','initial_dst_cell_id','current_dst_cell_id']].rename(columns={'dst_number':'sim','initial_dst_cell_id':'initial_cell','current_dst_cell_id':'current_cell'})
endpoints=pd.concat([source,destination],ignore_index=True).drop_duplicates(['sim','record_id']).sort_values(['sim','timestamp'])
endpoints['known_fraud']=endpoints.sim.isin(fraud)
endpoints['transition']=endpoints.initial_cell.astype(str)!=endpoints.current_cell.astype(str)

features=endpoints.groupby('sim').agg(unique_initial_cells=('initial_cell','nunique'),unique_current_cells=('current_cell','nunique'),cell_transition_events=('transition','sum'),activity_events=('record_id','nunique'),first_cell=('current_cell','first'),last_cell=('current_cell','last'),known_fraud=('known_fraud','max')).reset_index().rename(columns={'sim':'sim_number'})
features['cell_transition_rate']=features.cell_transition_events/features.activity_events.replace(0,np.nan)
features['cell_diversity_per_event']=features.unique_current_cells/features.activity_events.replace(0,np.nan)
features.to_csv(OUTPUT/'mobility_features.csv',index=False)
features.head()

,sim_number,unique_initial_cells,unique_current_cells,cell_transition_events,activity_events,first_cell,last_cell,known_fraud,cell_transition_rate,cell_diversity_per_event
0,+11102241212,1,1,0,1,0,0,False,0.0,1.0
1,+11124222602,1,1,0,1,0,0,False,0.0,1.0
2,+11185236570,1,1,0,1,0,0,False,0.0,1.0
3,+11201660070,1,1,0,1,0,0,False,0.0,1.0
4,+11278389544,1,1,0,1,0,0,False,0.0,1.0


In [3]:
# Count observed changes in current cell across chronological events.
endpoints['previous_cell']=endpoints.groupby('sim').current_cell.shift(1)
endpoints['cell_change']=endpoints['previous_cell'].notna() & (endpoints.current_cell.astype(str)!=endpoints.previous_cell.astype(str))
changes=endpoints.groupby('sim').cell_change.sum().rename('observed_cell_changes')
features=features.merge(changes,left_on='sim_number',right_index=True,how='left')
features['observed_cell_change_rate']=features.observed_cell_changes/features.activity_events.replace(0,np.nan)
features.to_csv(OUTPUT/'mobility_features.csv',index=False)

numeric=['unique_initial_cells','unique_current_cells','cell_transition_events','activity_events','cell_transition_rate','cell_diversity_per_event','observed_cell_changes','observed_cell_change_rate']
counts=features.known_fraud.value_counts().reindex([False,True],fill_value=0)
if (counts==0).any(): raise ValueError(f'Both classes are required; counts={counts.to_dict()}')
comparison=features.groupby('known_fraud',observed=False)[numeric].median().reindex([False,True]).T
comparison.columns=['known_legitimate_median','known_fraudulent_median']; comparison['difference']=comparison.known_fraudulent_median-comparison.known_legitimate_median
comparison.to_csv(OUTPUT/'mobility_group_median_comparison.csv')
print(comparison)

                           known_legitimate_median  known_fraudulent_median  \
unique_initial_cells                           1.0                 1.000000   
unique_current_cells                           1.0                 1.000000   
cell_transition_events                         0.0                 0.000000   
activity_events                                2.0                 3.000000   
cell_transition_rate                           0.0                 0.000000   
cell_diversity_per_event                       1.0                 0.333333   
observed_cell_changes                          1.0                 2.000000   
observed_cell_change_rate                      0.5                 0.666667   

                           difference  
unique_initial_cells         0.000000  
unique_current_cells         0.000000  
cell_transition_events       0.000000  
activity_events              1.000000  
cell_transition_rate         0.000000  
cell_diversity_per_event    -0.666667  
observed

In [4]:
# Identify mobility outliers relative to the legitimate population using IQR.
legit=features.loc[~features.known_fraud,numeric]; q1=legit.quantile(.25,numeric_only=True); q3=legit.quantile(.75,numeric_only=True); iqr=q3-q1
rows=[]
for _,row in features.iterrows():
    for col in numeric:
        if pd.isna(row[col]) or pd.isna(iqr[col]) or iqr[col]==0: continue
        if row[col]>q3[col]: rows.append({'sim_number':row.sim_number,'known_fraud':row.known_fraud,'feature':col,'value':row[col],'legitimate_q3':q3[col],'iqr_deviation':(row[col]-q3[col])/iqr[col],'direction':'high'})
        elif row[col]<q1[col]: rows.append({'sim_number':row.sim_number,'known_fraud':row.known_fraud,'feature':col,'value':row[col],'legitimate_q1':q1[col],'iqr_deviation':(q1[col]-row[col])/iqr[col],'direction':'low'})
pd.DataFrame(rows).sort_values('iqr_deviation',ascending=False).to_csv(OUTPUT/'mobility_iqr_deviations.csv',index=False)
print('mobility evidence rows:',len(rows))

mobility evidence rows: 6406


Interpretation rule: cell diversity or transitions are evidence of movement behavior, not physical distance. A lack of movement may also be meaningful for SIMBox behavior, so both high and low deviations must be retained.